# gramlab · core-v1 scoreboard

Runs the fixed benchmark of the stable core on Kaggle CPU (no GPU): role linker (learned vs rules vs gold), the ordered
meaning key, entity typing prior, WordNet coverage of layer 0, affect lexicon (if a VAD file is attached as a dataset),
and the index. Training text: AMALGUM (191k silver-parsed sentences, sparse-cloned); evaluation: gold GUM test.
Expected wall time ≈ 10 minutes. Outputs: `results/scoreboard.json`, `results/linker.pkl`.

See `gramlab/core/README.md` for the contracts and `RESULTS.md` run 9 for the baseline numbers.

In [ ]:
import os, sys, glob, subprocess, pathlib, json
WORK = pathlib.Path('/kaggle/working'); os.chdir(WORK)
def sh(c):
    print('$', c); r = subprocess.run(c, shell=True, text=True, capture_output=True)
    if r.returncode: print(r.stdout[-1500:], r.stderr[-1500:])
    return r.returncode == 0
sh('git clone --depth 1 https://github.com/furevych/gramlab.git gramlab')
sys.path.insert(0, str(WORK / 'gramlab'))
GUM = WORK / 'data' / 'gum'; GUM.mkdir(parents=True, exist_ok=True)
for split in ('train', 'dev', 'test'):
    sh(f'curl -sSL -o {GUM}/en_gum-ud-{split}.conllu https://raw.githubusercontent.com/UniversalDependencies/UD_English-GUM/master/en_gum-ud-{split}.conllu')
AMALGUM = 'data/amalgum/amalgum/*/dep/*.conllu'
if not glob.glob(AMALGUM):
    sh('git clone --depth 1 --filter=blob:none --sparse https://github.com/gucorpling/amalgum.git data/amalgum')
    sh('cd data/amalgum && git sparse-checkout set amalgum/academic/dep amalgum/bio/dep amalgum/fiction/dep amalgum/interview/dep amalgum/news/dep amalgum/voyage/dep amalgum/whow/dep')
print('GUM', sorted(p.name for p in GUM.glob('*.conllu')), '| AMALGUM files', len(glob.glob(AMALGUM)))
# layer-0 preset: WordNet through nltk (optional; the bench reports which backend it got)
sh('pip -q install nltk 2>/dev/null') 
try:
    import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)
    from nltk.corpus import wordnet as wn; print('WordNet ok:', [s.name() for s in wn.synsets('wash', pos='v')][:3])
except Exception as e:
    print('WordNet unavailable, lemma backend will be used:', e)
# affect lexicon: attach a Kaggle dataset containing NRC-VAD-Lexicon.txt (word, valence, arousal, dominance) to enable the affect row
VAD = next(iter(glob.glob('/kaggle/input/**/NRC-VAD-Lexicon*.txt', recursive=True) + glob.glob('/kaggle/input/**/*vad*.txt', recursive=True)), None)
print('VAD lexicon:', VAD)

In [ ]:
from gramlab.core import bench
board = bench.run(str(GUM), AMALGUM, VAD, out_dir=str(WORK / 'results'))

In [ ]:
# a few events through the pipeline, decoded
from gramlab.core import Pipeline, LearnedLinker, read_conllu
pipe = Pipeline(linker=LearnedLinker.load(str(WORK / 'results' / 'linker.pkl')))
test = read_conllu(str(GUM / 'en_gum-ud-test.conllu'))
shown = 0
for s in test:
    for e in pipe.events(s):
        if e.actor and e.patient and e.voice == 'passive':
            print(s.text()[:120]); print('   ', pipe.describe(e)); shown += 1
            break
    if shown >= 6: break
print(pipe.components())

## Reading the scoreboard
- **links**: the linker's F1 against UD gold; passive F1 and role swaps are the numbers that matter for the key.
- **keys**: `key_exact` = the class triple built from predicted roles equals the gold triple; `position_only_control`
  is what the toy map's rule gets on real text (near 0.7 on actives, ~0.08 on passives).
- **typing**: the prior every entity typer has to beat.
- **general**: with WordNet, the share of actions / arguments that got a concept id and how many families the verbs collapse into at depth 2.
- A component change is accepted when its own row does not drop and nothing else moves.